# Dummy XGBoost smoke test

End-to-end check of `ModelTrainingWorkflow`: 1000-row parquet, temporal train/validation/test split, a small XGBoost model.

Tracking goes to the Docker MLflow server (`http://localhost:5000`). That server proxies artifacts into the MinIO bucket `s3://mlflow/` (`http://localhost:9000`).

In [ ]:
from pathlib import Path
import os

import mlflow
import numpy as np
import pandas as pd

from modeling_fraud_system.run_time_configuration import BaseConfigParams
from modeling_fraud_system.training import ModelTrainingWorkflow

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "runs_entrypoints":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DUMMY_PATH = DATA_DIR / "dummy_fraud_1000.parquet"

TARGET_COLUMN = "Flag_NotPaid_Within_90_Days_After_DueDate"
N_ROWS = 1000
N_TRAIN_PERIOD = 800
N_TEST_PERIOD = 200
SEED = 42


def _load_env(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


_load_env(PROJECT_ROOT / ".env")



MLFLOW_TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dummy parquet: {DUMMY_PATH}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

Project root: /home/martinezyamamotoarthur/modeling_fraud_system
Dummy parquet: /home/martinezyamamotoarthur/modeling_fraud_system/data/dummy_fraud_1000.parquet
MLflow tracking URI: http://localhost:5000


In [3]:
rng = np.random.default_rng(SEED)

train_start = pd.Timestamp("2024-01-01", tz="UTC")
train_end = pd.Timestamp("2025-06-30 23:59:59", tz="UTC")
test_start = pd.Timestamp("2025-07-01", tz="UTC")
test_end = pd.Timestamp("2025-08-31 23:59:59", tz="UTC")

train_offsets = rng.integers(0, int((train_end - train_start).total_seconds()) + 1, size=N_TRAIN_PERIOD)
test_offsets = rng.integers(0, int((test_end - test_start).total_seconds()) + 1, size=N_TEST_PERIOD)
request_datetime = pd.concat(
    [
        pd.Series(train_start + pd.to_timedelta(train_offsets, unit="s")),
        pd.Series(test_start + pd.to_timedelta(test_offsets, unit="s")),
    ],
    ignore_index=True,
)

transaction_amt = rng.lognormal(mean=3.5, sigma=0.8, size=N_ROWS)
account_age_days = rng.integers(1, 2000, size=N_ROWS).astype(float)
n_prev_transactions = rng.poisson(8, size=N_ROWS).astype(float)
distance = rng.exponential(50, size=N_ROWS)
hour_of_day = rng.integers(0, 24, size=N_ROWS).astype(float)
n_failed_attempts = rng.poisson(0.4, size=N_ROWS).astype(float)
email_risk_score = rng.beta(2, 8, size=N_ROWS)
device_mismatch = rng.binomial(1, 0.15, size=N_ROWS).astype(float)
product_cd = rng.choice(["W", "C", "H", "R", "S"], size=N_ROWS)

logit = (
    -2.2
    + 0.02 * (transaction_amt - 30)
    - 0.001 * account_age_days
    + 0.45 * n_failed_attempts
    + 2.2 * email_risk_score
    + 0.9 * device_mismatch
    + 0.004 * distance
    + 0.35 * (hour_of_day < 6)
)
prob = 1.0 / (1.0 + np.exp(-logit))
y = rng.binomial(1, np.clip(prob, 0.03, 0.55), size=N_ROWS)

dummy_df = pd.DataFrame(
    {
        "RequestDateTime": request_datetime,
        "transaction_amt": transaction_amt,
        "account_age_days": account_age_days,
        "n_prev_transactions": n_prev_transactions,
        "distance": distance,
        "hour_of_day": hour_of_day,
        "n_failed_attempts": n_failed_attempts,
        "email_risk_score": email_risk_score,
        "device_mismatch": device_mismatch,
        "ProductCD": product_cd,
        TARGET_COLUMN: y,
    }
).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

dummy_df.to_parquet(DUMMY_PATH, engine="pyarrow", index=False)

print(dummy_df.shape)
print(dummy_df.dtypes)
print(dummy_df[TARGET_COLUMN].value_counts(normalize=True).round(3))
print(
    dummy_df.assign(period=np.where(dummy_df["RequestDateTime"] < test_start, "train_val", "test"))
    .groupby("period")
    .size()
)
dummy_df.head()

(1000, 11)
RequestDateTime                              datetime64[us, UTC]
transaction_amt                                          float64
account_age_days                                         float64
n_prev_transactions                                      float64
distance                                                 float64
hour_of_day                                              float64
n_failed_attempts                                        float64
email_risk_score                                         float64
device_mismatch                                          float64
ProductCD                                                    str
Flag_NotPaid_Within_90_Days_After_DueDate                  int64
dtype: object
Flag_NotPaid_Within_90_Days_After_DueDate
0    0.822
1    0.178
Name: proportion, dtype: float64
period
test         200
train_val    800
dtype: int64


,RequestDateTime,transaction_amt,account_age_days,n_prev_transactions,distance,hour_of_day,n_failed_attempts,email_risk_score,device_mismatch,ProductCD,Flag_NotPaid_Within_90_Days_After_DueDate
0,2024-11-23 09:53:27+00:00,60.642655,1372.0,3.0,10.445285,12.0,0.0,0.076612,0.0,R,0
1,2024-03-27 05:07:32+00:00,16.107458,354.0,5.0,272.628554,13.0,1.0,0.166671,0.0,W,0
2,2024-02-17 04:24:20+00:00,20.877712,1783.0,8.0,98.915776,6.0,0.0,0.105622,0.0,S,0
3,2024-12-02 01:34:38+00:00,24.351739,1451.0,7.0,71.949255,2.0,0.0,0.065778,0.0,S,0
4,2024-10-03 23:05:57+00:00,48.574584,1499.0,8.0,2.129837,11.0,0.0,0.087183,0.0,H,0


In [4]:
runtime_params = BaseConfigParams(
    experiment_name="dummy_fraud_xgboost",
    mlflow_run_name="dummy_xgb_simple",
    training_start_date="2024-01-01",
    training_end_date="2025-06-30",
    test_start_date="2025-07-01",
    test_end_date="2025-08-31",
    validation_start_date="2024-01-01",
    validation_end_date="2025-06-30",
)

workflow = ModelTrainingWorkflow(runtime_params)
workflow.load_data(path_external_data=str(DUMMY_PATH), columns=list(dummy_df.columns))
workflow.split_data(target_column=TARGET_COLUMN, validation_split=0.2)
workflow.generate_preprocessed_data(numeric_only=True)

print("train", workflow.x_train_preproc.shape, "pos_rate", float(workflow.y_train.mean()))
print("valid", workflow.x_validation_preproc.shape, "pos_rate", float(workflow.y_validation.mean()))
print("test ", workflow.x_test_preproc.shape, "pos_rate", float(workflow.y_test.mean()))
print("numeric features", list(workflow.x_train_preproc.columns))

2026-09-20 16:25:02,579 - modeling_fraud_system.training - INFO - Initialized the SolvencyNewCustomersModelTraining workflow.
2026-09-20 16:25:02,600 - modeling_fraud_system.training - INFO - Splitting data - Training+Validation period: 2024-01-01 00:00:00+00:00 to 2025-06-30 23:59:59+00:00 (random split with 20% validation), Test period: 2025-07-01 00:00:00+00:00 to 2025-08-31 23:59:59+00:00
2026-09-20 16:25:02,622 - modeling_fraud_system.training - INFO - Data split sizes - Training: 640, Validation: 160, Testing: 200
2026-09-20 16:25:02,625 - modeling_fraud_system.training - INFO - 
Target Rate Distribution:
2026-09-20 16:25:02,626 - modeling_fraud_system.training - INFO -   Training set: 16.88% (positive class)
2026-09-20 16:25:02,627 - modeling_fraud_system.training - INFO -   Test set: 21.50% (positive class)
2026-09-20 16:25:02,628 - modeling_fraud_system.training - INFO -   Validation set: 16.88% (positive class)
2026-09-20 16:25:02,628 - modeling_fraud_system.training - INFO -

train (640, 8) pos_rate 0.16875
valid (160, 8) pos_rate 0.16875
test  (200, 8) pos_rate 0.215
numeric features ['transaction_amt', 'account_age_days', 'n_prev_transactions', 'distance', 'hour_of_day', 'n_failed_attempts', 'email_risk_score', 'device_mismatch']


In [5]:
results = workflow.train_model(
    hyperparameters={
        "n_estimators": 40,
        "max_depth": 3,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    model_type="xgboost",
    log_into_mlflow=True,
    show_plots=False,
    verbose=False,
    seed=SEED,
    run_name="dummy_xgb_simple",
)

metrics_table = pd.DataFrame(
    {
        "train": results["train_metrics"],
        "validation": results["val_metrics"],
        "test": results["test_metrics"],
    }
).loc[["accuracy", "precision", "recall", "f1", "auc", "pr_auc", "ks_statistic"]]

run_id = workflow.run_time_config.mlflow_run_id
client = mlflow.tracking.MlflowClient()
run = client.get_run(run_id)

print("threshold", round(results["threshold"], 4))
print("mlflow run id", run_id)
print("tracking uri", mlflow.get_tracking_uri())
print("artifact uri", run.info.artifact_uri)
print("logged artifacts", [a.path for a in client.list_artifacts(run_id)])
metrics_table.round(4)

2026-09-20 16:25:02,649 - modeling_fraud_system.training - INFO - Starting model training without hyperparameter tuning using xgboost.
2026-09-20 16:25:02,651 - modeling_fraud_system.training - INFO - Training xgboost model with fixed hyperparameters.
/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [16:25:02] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026-09-20 16:25:02,733 - modeling_fraud_system.training - INFO - Calculated optimal threshold (F1-based) on validation set: 0.2559
2026-09-20 16:25:02,753 - modeling_fraud_system.evaluation - INFO - Training Metrics - AUC: 0.9087, PR-AUC: 0.7210, KS: 0.6740
2026-09-20 16:25:02,763 - modeling_fraud_system.evaluation - INFO - Validation Metrics - AUC: 0.6597, PR-AUC: 0.3558, KS: 0.4052
2026-09-20 16:25:02,776 - modeling_fraud_system.evaluation - INFO - Test Metric

🏃 View run dummy_xgb_simple at: http://localhost:5000/#/experiments/1/runs/cb8596ccd88e4d089b1d55a61d1db2fc
🧪 View experiment at: http://localhost:5000/#/experiments/1
threshold 0.2559
mlflow run id cb8596ccd88e4d089b1d55a61d1db2fc
tracking uri http://localhost:5000
artifact uri mlflow-artifacts:/1/cb8596ccd88e4d089b1d55a61d1db2fc/artifacts
logged artifacts ['plots']


,train,validation,test
accuracy,0.885938,0.8,0.79
precision,0.669903,0.428571,0.512195
recall,0.638889,0.555556,0.488372
f1,0.654028,0.483871,0.5
auc,0.908661,0.659705,0.766405
pr_auc,0.721023,0.355771,0.488329
ks_statistic,0.673977,0.40518,0.439046
